<a href="https://colab.research.google.com/github/alanmathew184-hub/python-project/blob/main/project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
class Room:
    def __init__(self, room_number, capacity):
        if not isinstance(room_number, str) or not room_number:
            raise ValueError("Room number must be a non-empty string.")
        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError("Capacity must be a positive integer.")
        self.room_number = room_number
        self.capacity = capacity
        self.occupants = []

    def add_occupant(self, student_id):
        if not isinstance(student_id, str) or not student_id:
            raise ValueError("Student ID must be a non-empty string.")
        if len(self.occupants) < self.capacity:
            if student_id not in self.occupants:
                self.occupants.append(student_id)
                return True
        return False

    def remove_occupant(self, student_id):
        if not isinstance(student_id, str) or not student_id:
            raise ValueError("Student ID must be a non-empty string.")
        if student_id in self.occupants:
            self.occupants.remove(student_id)
            return True
        return False

    def is_full(self):
        return len(self.occupants) == self.capacity

    def is_available(self):
        return len(self.occupants) < self.capacity

    def __repr__(self):
        return f"Room(Number={self.room_number}, Capacity={self.capacity}, Occupants={len(self.occupants)}/{self.capacity})"

class Student:
    def __init__(self, student_id, name, year, gender=None):
        if not isinstance(student_id, str) or not student_id:
            raise ValueError("Student ID must be a non-empty string.")
        if not isinstance(name, str) or not name:
            raise ValueError("Student name must be a non-empty string.")
        if not isinstance(year, int) or year <= 0:
            raise ValueError("Year must be a positive integer.")
        if gender is not None and gender not in ['Male', 'Female', 'Other']:
            raise ValueError("Gender must be 'Male', 'Female', 'Other', or None.")
        self.student_id = student_id
        self.name = name
        self.year = year
        self.gender = gender
        self.allocated_room_number = None

    def __repr__(self):
        return f"Student(ID={self.student_id}, Name={self.name}, Year={self.year}, Room={self.allocated_room_number})"

class HostelManagementSystem:
    def __init__(self):
        self.rooms = {}
        self.students = {}

    def add_room(self, room_number, capacity):
        if room_number in self.rooms:
            return False
        try:
            room = Room(room_number, capacity)
            self.rooms[room_number] = room
            return True
        except ValueError:
            return False

    def register_student(self, student_id, name, year, gender=None):
        if student_id in self.students:
            return False
        try:
            student = Student(student_id, name, year, gender)
            self.students[student_id] = student
            return True
        except ValueError:
            return False

    def allocate_room(self, student_id, room_number):
        student = self.students.get(student_id)
        room = self.rooms.get(room_number)

        if not student: return False
        if not room: return False
        if student.allocated_room_number: return False

        if room.add_occupant(student_id):
            student.allocated_room_number = room_number
            return True
        return False

    def vacate_room(self, student_id):
        student = self.students.get(student_id)
        if not student: return False
        if not student.allocated_room_number: return False

        room_number = student.allocated_room_number
        room = self.rooms.get(room_number)

        if room and room.remove_occupant(student_id):
            student.allocated_room_number = None
            return True
        return False

    def get_room_occupancy(self, room_number):
        room = self.rooms.get(room_number)
        if not room: return None

        occupant_details = []
        for student_id in room.occupants:
            student = self.students.get(student_id)
            if student:
                occupant_details.append({'id': student.student_id, 'name': student.name})

        return {
            'room_number': room.room_number,
            'capacity': room.capacity,
            'current_occupancy': len(room.occupants),
            'occupants': occupant_details
        }

    def generate_allocation_report(self):
        report = []
        for room_number, room in self.rooms.items():
            occupants_info = []
            for student_id in room.occupants:
                student = self.students.get(student_id)
                if student:
                    occupants_info.append(f"{student.name} (ID: {student.student_id})")

            report.append({
                'room_number': room.room_number,
                'capacity': room.capacity,
                'occupied_slots': len(room.occupants),
                'available_slots': room.capacity - len(room.occupants),
                'occupants': ', '.join(occupants_info) if occupants_info else 'None'
            })
        return report



hostel_system = HostelManagementSystem()

print("\n--- Adding Rooms ---")
if hostel_system.add_room("A101", 2): print("Added Room A101")
else: print("Failed to add Room A101")
if hostel_system.add_room("A102", 1): print("Added Room A102")
else: print("Failed to add Room A102")
if not hostel_system.add_room("A101", 2): print("Error: A101 already exists (expected). Failed to add A101 again.")

print("\n--- Registered Rooms ---")
for room_num, room_obj in hostel_system.rooms.items():
    print(room_obj)

print("\n--- Registering Students ---")
if hostel_system.register_student("S001", "Alice Smith", 1, "Female"): print("Registered S001 Alice")
if hostel_system.register_student("S002", "Bob Johnson", 1, "Male"): print("Registered S002 Bob")
if hostel_system.register_student("S003", "Charlie Brown", 2, "Male"): print("Registered S003 Charlie")
if not hostel_system.register_student("S001", "Alice Smith Duplicate", 1): print("Error: S001 already registered (expected). Failed to register S001 again.")

print("\n--- Registered Students (before allocation) ---")
for student_id, student_obj in hostel_system.students.items():
    print(student_obj)

print("\n--- Allocating Students to Rooms ---")
if hostel_system.allocate_room("S001", "A101"): print("Allocated S001 to A101")
else: print("Failed to allocate S001 to A101")
if hostel_system.allocate_room("S002", "A101"): print("Allocated S002 to A101")
else: print("Failed to allocate S002 to A101")
if hostel_system.allocate_room("S003", "A102"): print("Allocated S003 to A102")
else: print("Failed to allocate S003 to A102")

print("\n--- Attempting invalid allocations ---")
if not hostel_system.allocate_room("S001", "A102"): print("Failed to allocate S001 to A102 (expected: Alice is already in A101)")
if not hostel_system.allocate_room("S004", "A101"): print("Failed to allocate S004 to A101 (expected: Student S004 does not exist)")
if hostel_system.register_student("S004", "Diana Prince", 2, "Female"): print("Registered S004 Diana")
if not hostel_system.allocate_room("S004", "A101"): print("Failed to allocate S004 to A101 (expected: Room A101 is full)")

print("\n--- Checking Room Occupancy for A101 ---")
occupancy_a101 = hostel_system.get_room_occupancy("A101")
if occupancy_a101:
    print(f"Room {occupancy_a101['room_number']}: {occupancy_a101['current_occupancy']}/{occupancy_a101['capacity']} occupants")
    for occupant in occupancy_a101['occupants']:
        print(f"  - {occupant['name']} (ID: {occupant['id']})")

print("\n--- Vacating a Room (S001 from A101) ---")
if hostel_system.vacate_room("S001"): print("S001 vacated successfully.")
else: print("Failed to vacate S001.")

print("\n--- Checking Room Occupancy for A101 after vacating ---")
occupancy_a101_after_vacate = hostel_system.get_room_occupancy("A101")
if occupancy_a101_after_vacate:
    print(f"Room {occupancy_a101_after_vacate['room_number']}: {occupancy_a101_after_vacate['current_occupancy']}/{occupancy_a101_after_vacate['capacity']} occupants")
    if occupancy_a101_after_vacate['occupants']:
        for occupant in occupancy_a101_after_vacate['occupants']:
            print(f"  - {occupant['name']} (ID: {occupant['id']})")
    else:
        print("  (No occupants)")

print("\n--- Generating Allocation Report ---")
report = hostel_system.generate_allocation_report()
for room_data in report:
    print(f"Room {room_data['room_number']}: Occupied {room_data['occupied_slots']}/{room_data['capacity']} (Available: {room_data['available_slots']}) - Occupants: {room_data['occupants']}")

print("\n--- Updated Student Information ---")
for student_id, student_obj in hostel_system.students.items():
    print(student_obj)


--- Adding Rooms ---
Added Room A101
Added Room A102
Error: A101 already exists (expected). Failed to add A101 again.

--- Registered Rooms ---
Room(Number=A101, Capacity=2, Occupants=0/2)
Room(Number=A102, Capacity=1, Occupants=0/1)

--- Registering Students ---
Registered S001 Alice
Registered S002 Bob
Registered S003 Charlie
Error: S001 already registered (expected). Failed to register S001 again.

--- Registered Students (before allocation) ---
Student(ID=S001, Name=Alice Smith, Year=1, Room=None)
Student(ID=S002, Name=Bob Johnson, Year=1, Room=None)
Student(ID=S003, Name=Charlie Brown, Year=2, Room=None)

--- Allocating Students to Rooms ---
Allocated S001 to A101
Allocated S002 to A101
Allocated S003 to A102

--- Attempting invalid allocations ---
Failed to allocate S001 to A102 (expected: Alice is already in A101)
Failed to allocate S004 to A101 (expected: Student S004 does not exist)
Registered S004 Diana
Failed to allocate S004 to A101 (expected: Room A101 is full)

--- Check